## Music Generation Using CNN
dataset - midi(Music Dataset) https://www.kaggle.com/datasets/kritanjalijain/music-midi-dataset

In [30]:
import numpy as np
from music21 import converter, chord, instrument
import tensorflow as tf

In [31]:
import glob
midi_files= glob.glob("/Users/jellyfish/DeveloperLocal/DataScience2026/Music_CNN/archive/midi_dataset/**.mid")
print(len(midi_files))

50


In [32]:
from music21 import converter, instrument, chord
from music21 import note as m21_note  

notes = []
for file in midi_files:
    try:
        midi = converter.parse(file)
        parts = instrument.partitionByInstrument(midi)
        
        if parts and len(parts.parts) > 0:
            notes_to_parse = parts.parts[0].recurse()
        else:
            notes_to_parse = midi.flat.notes
    
        for element in notes_to_parse:
            if isinstance(element, m21_note.Note):
                notes.append(str(element.pitch))
            elif isinstance(element, chord.Chord):
                notes.append('.'.join(str(n) for n in element.normalOrder)) 

    except Exception as e:
        print(f"error reading {file}: {e}")
        
print("total notes: ", len(notes))
print(notes[:20])

total notes:  3892
['E6', 'E6', 'E-6', 'C#6', 'E6', 'E-6', 'E-6', 'C#6', 'B5', 'B5', 'C#6', 'C#6', 'B5', 'A5', 'B5', 'B5', 'A5', 'G#5', 'A5', 'A5']


In [33]:
pitchnames = sorted(set(notes))
print("total unique notes: ", len(pitchnames))

total unique notes:  113


In [34]:
notes_to_int = {note: number for number, note in enumerate(pitchnames)}
print(f"vocabulary size: {len(notes_to_int)}")

vocabulary size: 113


In [35]:
sequence_length = 100
network_input = []
network_output = []
for i in range(len(notes) - sequence_length):
    sequence_in = notes[i:i + sequence_length]
    sequence_out = notes[i + sequence_length]
    network_input.append([notes_to_int[n] for n in sequence_in])
    network_output.append(notes_to_int[sequence_out])

print("Input sequence : ", len(network_input))
print("Output sequence : ", len(network_output))




Input sequence :  3792
Output sequence :  3792


In [36]:
n_pattern = len(network_input)
network_input = np.reshape(
    network_input,
    (n_pattern, sequence_length, 1)
)
network_input = network_input / float(len(pitchnames))
print("Final Reshaped Shape: ", network_input.shape)

Final Reshaped Shape:  (3792, 100, 1)


In [37]:
# import os
# import shutil

# source_dir = "/Users/jellyfish/Downloads/archive"
# dest_dir = "/Users/jellyfish/DeveloperLocal/DataScience2026/music_dataset"

# os.makedirs(dest_dir, exist_ok=True)

# for root, dirs, files in os.walk(source_dir):
#     for file in files:
#         if file.lower().endswith('.mid'):
#             source_file = os.path.join(root, file)
#             dest_file = os.path.join(dest_dir, file)
            
#             # Handle duplicate file names by adding a counter
#             name, ext = os.path.splitext(file)
#             counter = 1
#             while os.path.exists(dest_file):
#                 dest_file = os.path.join(dest_dir, f"{name}_{counter}{ext}")
#                 counter += 1
                
#             shutil.copy2(source_file, dest_file)

# print("Finished! All files copied safely.")


In [38]:
network_output  = tf.keras.utils.to_categorical(network_output)
print(network_output)

[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]


In [39]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dropout, Dense
model = Sequential()
model.add(LSTM(
    512,
    input_shape = (network_input.shape[1], network_input.shape[2]),
    return_sequences=True
))

model.add(Dropout(0.3))

model.add(LSTM(
    512,
    return_sequences=True,
    ))

model.add(Dropout(0.3))

model.add(LSTM(512))

model.add(Dense(256, activation='relu'))

model.add(Dropout(0.3))

model.add(Dense(network_output.shape[1], activation='softmax'))

model.compile(
    loss='categorical_crossentropy',
    optimizer = 'adam'
)

model.summary()


/opt/anaconda3/lib/python3.13/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_7 (LSTM)                   │ (None, 100, 512)       │     1,052,672 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 100, 512)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_8 (LSTM)                   │ (None, 100, 512)       │     2,099,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 100, 512)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_9 (LSTM)                   │ (None, 512)            │     2,099,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 113)            │        29,041 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,411,441 (20.64 MB)

 Trainable params: 5,411,441 (20.64 MB)

 Non-trainable params: 0 (0.00 B)

In [46]:
from tensorflow.keras.callbacks import ModelCheckpoint
checkpoint = ModelCheckpoint(
    "best.model.keras",
    monitor = 'loss',
    verbose = 1,
    save_best_only=True,
    mode='min'
)

model.fit(
    network_input,
    network_output,
    epochs = 5,
    batch_size = 64,
    callbacks = [checkpoint]
)

Epoch 1/5
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - loss: 4.4444
Epoch 1: loss improved from None to 4.33468, saving model to best.model.keras

Epoch 1: finished saving model to best.model.keras
60/60 ━━━━━━━━━━━━━━━━━━━━ 63s 1s/step - loss: 4.3347
Epoch 2/5
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - loss: 4.1416
Epoch 2: loss improved from 4.33468 to 4.05677, saving model to best.model.keras

Epoch 2: finished saving model to best.model.keras
60/60 ━━━━━━━━━━━━━━━━━━━━ 64s 1s/step - loss: 4.0568
Epoch 3/5
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - loss: 3.8659
Epoch 3: loss improved from 4.05677 to 3.84046, saving model to best.model.keras

Epoch 3: finished saving model to best.model.keras
60/60 ━━━━━━━━━━━━━━━━━━━━ 68s 1s/step - loss: 3.8405
Epoch 4/5
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - loss: 3.6932
Epoch 4: loss improved from 3.84046 to 3.65256, saving model to best.model.keras

Epoch 4: finished saving model to best.model.keras
60/60 ━━━━━━━━━━━━━━━━━━━━ 69s 1s/step - loss: 3.6526
Epo

In [49]:
from tensorflow.keras.models import load_model
import random
model = load_model("best.model.keras")
start = random.randint(0, len(network_input)-1)

int_to_note = {number: note for number, note in enumerate(pitchnames)}

pattern = network_input[start]
prediction_output = []

for _ in range (500):
    prediction_input = np.reshape(pattern, (1, len(pattern), 1))
    prediction = model.predict(prediction_input, verbose=0)
    index = np.argmax(prediction)
    result = int_to_note = [index]
    prediction_output.append(result)
    pattern = np.append(pattern, index/float(len(pitchnames)))
    pattern = pattern[1:]



In [ ]:
from music21 import note, chord, instrument, stream

offset = 0
output_notes = []
for pattern in prediction_output:
    if isinstance(pattern, list):
        chord_notes = []
        for current_note in pattern:
            if str(current_note).isdigit():
                new_note = note.Note(int(current_note))
            else:
                new_note = note.Note(str(current_note))
            new_note.storedInstrument = instrument.Piano()
            chord_notes.append(new_note)
        new_chord = chord.Chord(chord_notes)
        new_chord.offset = offset
        output_notes.append(new_chord)
    elif "." in str(pattern) or str(pattern).isdigit():
        notes_in_chord = str(pattern).split(".")
        chord_notes = []
        for current_note in notes_in_chord:
            new_note = note.Note(int(current_note))
            new_note.storedInstrument = instrument.Piano()
            chord_notes.append(new_note)
        new_chord = chord.Chord(chord_notes)
        new_chord.offset = offset
        output_notes.append(new_chord)
    else:
        new_note = note.Note(pattern)
        new_note.offset = offset
        new_note.storedInstrument = instrument.Piano()
        output_notes.append(new_note)
    offset += 0.5

midi_stream = stream.Stream(output_notes)
midi_stream.write("midi", fp="generate_music.mid")
print("The Project have been completed and the music has been generated - welldone")


hi


In [59]:
import pygame
import time

pygame.init()
pygame.mixer.init()
pygame.mixer.music.load("generate_music.mid")
pygame.mixer.music.play()

while pygame.mixer.music.get_busy():
    time.sleep(1)

print("music has been finished")


AttributeError: partially initialized module 'pygame' from '/opt/anaconda3/lib/python3.13/site-packages/pygame/__init__.py' has no attribute 'color' (most likely due to a circular import)